In [ ]:
from fem import (
    GMSHtools,
    FEMModel,
    plot_gmsh_mesh,
    globalParameters,
)
import os
import numpy as np
import matplotlib.pyplot as plt
import gmsh
np.set_printoptions(suppress=True, precision=6, linewidth=400)


In [ ]:
globalParameters['nDoF'] = 3
globalParameters['nDIM'] = 3

In [ ]:
# General model parameters
output_file = 'cantiliver_beam.msh'

In [ ]:
# read mesh — node_map and system_nDof auto-generated
mesh = GMSHtools(output_file)

In [ ]:
# plot_gmsh_mesh(mesh,
#                show_node_labels    = False,
#                show_element_labels = False,
#                show_node_points    = False,
#                view_3d             = True, elev=45, azim=-45,
#                figsize             = (12, 8))

In [ ]:

load_dictionary = {
                31:   {'value': 1, 'direction': '-z'},     
}

In [ ]:

# load_dictionary = {
#                 27:   {'value': 1, 'direction': 'z'},     
#                 28:   {'value': 1, 'direction': '-z'},     
#                 29:   {'value': 1, 'direction': '-y'},   
#                 30:   {'value': 1, 'direction': 'y'},   
# }

In [ ]:
model = FEMModel(
    mesh                = mesh,
    section_dictionary  = {},
    restrain_dictionary = {},
    load_dictionary     = {},
    element_class_map   = None,       # OpenSees only
    analysis_type       = '3D',
    consistent_loads    = False,
)

In [ ]:
# Mesh diagnostics
model.check_mesh()

## Opensees

In [ ]:
import openseespy.opensees as ops

# import sys
# sys.path.insert(0, r"C:\Dropbox\01. Brain\11. GitHub\OpenSees\compile_windows\build-win11")

# import opensees as ops
# print(ops.__file__)



import opsvis as opsv

ops.wipe()
ops.model('basicBuilder','-ndm',2,'-ndf',2)

In [ ]:
# Nodes
for tag, (x, y, z) in mesh.nodes.items():
    ops.node(tag, x, y , z)

In [ ]:
# Boundary conditions
fixed_nodes = set()
for tag in mesh.physical_groups['support'].nodes:
    if tag not in fixed_nodes:
        fixed_nodes.add(tag)
        # ops.fix(tag, 1, 1, 1, 1, 1, 1)
        ops.fix(tag, 1, 1, 1)

In [ ]:
# Material
E = 200000e10
nu = 0.30 
rho = 7.85e-9  
ops.nDMaterial('ElasticIsotropic', 1, E, nu, rho)

In [ ]:
# Elements
group = mesh.physical_groups['solid'].elements
for elem_tag, conn in zip(group['element_tags'], group['connectivity']):
    n1, n2, n3, n4 = conn
    ops.element('FourNodeTetrahedron', elem_tag, n1, n2, n3, n4, 1)

In [ ]:
# ops.timeSeries('Linear', 1)
# ops.pattern('Plain', 1, 1)

# for tag, force in model.F_nodal.items():
#     if np.any(np.abs(force) > 0):
#         ops.load(tag, *force.tolist())

In [ ]:
# Twist parameters — rotation around X axis, section in YZ plane
cy    = 150.0           
cz    = 150.0           
theta = np.radians(45)  

# Prescribed displacements 
ops.timeSeries('Linear', 1)
ops.pattern('Plain', 1, 1)

for tag in mesh.physical_groups['load'].nodes:
    x, y, z = mesh.nodes[tag]
    dy = (y - cy) * np.cos(theta) - (z - cz) * np.sin(theta) - (y - cy)
    dz = (y - cy) * np.sin(theta) + (z - cz) * np.cos(theta) - (z - cz)
    ops.sp(tag, 1, 0.0)  
    ops.sp(tag, 2, dy)
    ops.sp(tag, 3, dz )

In [ ]:
NstepGravity = 10
DGravity     = 1 / NstepGravity

ops.system("SparseSYM")
ops.numberer("RCM") 
ops.constraints('Transformation')
ops.integrator("LoadControl", DGravity)
ops.test("NormUnbalance", 1.0e-3, 100, 0)
ops.algorithm("Newton")
ops.analysis("Static")

for step in range(NstepGravity):
    ops.analyze(1)
    model.set_results_opensees(ops, step=step)

# ops.analyze(NstepGravity)
# # save last step
# model.set_results_opensees(ops, step=0, time=1.0)

In [ ]:
# Static results in gmsh — last step
model.plot2gmsh(
    step           = -1,
    source         = 'opensees',
    disp_factor    = 1,
    show_disp      = True,
    show_loads     = True,
    show_reactions = True,
    show_stress    = True,
    show_strain    = True,
    show_vm        = True,
    show_averaged  = True,
)

In [ ]:
# Animate displacements in gmsh
model.plot2gmsh_animate(disp_factor=1)